# 📘 第7周 · Day2 — 工具编排与自动化

> **LLM工程实战（OpenClaw）** · 2026年7月14日
>
> 🎯 **今日目标**：掌握从单个工具调用到复杂自动化工作流的设计方法

---

## 📊 学习进度

| 阶段 | 状态 | 说明 |
|------|------|------|
| W1-W2 Transformer & 训练 | ✅ 已完成 | 理论基础扎实 |
| W3 推理优化 | ✅ 已完成 | 量化、投机采样等 |
| W4 RAG系统 | ✅ 已完成 | 检索增强生成 |
| W5 Function Calling | ✅ 已完成 | 工具调用理论 |
| W6 Agent框架 | ✅ 已完成 | 推理-行动循环 |
| **W7 Day1 System Prompt** | ✅ 已完成 | **提示词工程** |
| **W7 Day2 工具编排与自动化** | 🔥 今天 | **本节内容** |
| W7 Day3 多Agent协作 | ⬜ 明天 | Agent间协调 |

**本周总进度**：█░░░░░░░░░ 1/3 完成（33%）

---

## 📋 今日学习路线

```
08:00-09:00  ┃ 📖 理论：从Function Calling到工具编排
09:00-10:00  ┃ 📖 理论：定时任务（Cron）与调度
10:00-11:00  ┃ 📖 理论：跨平台消息路由
11:00-12:00  ┃ 💻 实践：工作流设计模式
     午休
13:00-14:00  ┃ 📖 理论：事件驱动架构
14:00-15:00  ┃ 💻 实践：Webhook与API集成
15:00-16:00  ┃ 🏗️ 综合案例：每日AI新闻推送工作流
16:00-17:00  ┃ ✏️ 练习与课后测试
```


---

## 🔄 往期回顾

### 一、W5 Function Calling — 理论回顾

还记得W5学的Function Calling吗？快速回顾核心链路：

| 概念 | 核心要点 |
|------|----------|
| **工具声明** | JSON Schema描述函数签名（名称、参数、类型） |
| **工具选择** | LLM根据用户意图，决定调用哪个工具 |
| **参数填充** | LLM从自然语言中提取结构化参数 |
| **执行与返回** | 外部系统执行，结果返回给LLM |
| **响应生成** | LLM基于工具结果生成最终回答 |

**W5学到的是「单次调用」——一个用户请求 → 一个工具调用 → 一个响应。**

今天我们要学的，是把这种模式放大到：

> **多工具串联 → 自动调度 → 跨平台 → 事件驱动 → 持续运行**

这就是**工具编排（Tool Orchestration）**。

---

### 二、Day1 System Prompt — 关联回顾

昨天我们学了System Prompt工程，它和今天的工具编排有什么关系？

```
┌─────────────────────────────────────────────┐
│           System Prompt（Day1）              │
│  "你是助手，可以调用天气API、搜索、日历..."    │
│  "收到消息时，先判断意图，再选择工具"          │
│  "如果是定时任务，使用Cron调度器"             │
└──────────────────┬──────────────────────────┘
                   │ 定义能力与行为规则
                   ▼
┌─────────────────────────────────────────────┐
│        工具编排（Day2 今天）                  │
│  工具声明 + 调度器 + 路由器 + 事件总线        │
│  Cron定时 + Webhook触发 + 跨平台推送         │
└─────────────────────────────────────────────┘
```

**System Prompt定义"做什么"，工具编排定义"怎么做"。**


---

## 1️⃣ 从 Function Calling 到工具编排

### 1.1 单工具 vs 工具编排

**W5的单工具调用模式：**

```
用户消息 → [LLM] → 选择工具 → 执行 → 结果回传 → [LLM] → 回复用户
```

**Day2的工具编排模式：**

```
触发源（定时/事件/消息）
    │
    ▼
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│ 工具A    │───▶│ 工具B    │───▶│ LLM判断  │───▶│ 工具C    │
│ (搜索)   │    │ (提取)   │    │ (下一步) │    │ (推送)   │
└──────────┘    └──────────┘    └──────────┘    └──────────┘
                                                        │
                    ┌───────────────────────────────────┘
                    ▼
              ┌──────────┐
              │ 响应     │
              │ (多平台) │
              └──────────┘
```

### 1.2 工具编排的五个层次

| 层次 | 名称 | 描述 | 例子 |
|------|------|------|------|
| L0 | **单工具** | 一个请求调用一个工具 | "查天气"→调用weather API |
| L1 | **顺序链** | 工具A的结果传给工具B | 搜索→提取→总结 |
| L2 | **条件分支** | 根据LLM判断走不同路径 | 如果紧急→短信，否则→邮件 |
| L3 | **并行执行** | 多个工具同时运行，合并结果 | 同时查天气+日历+新闻 |
| L4 | **事件循环** | 持续监听、自动响应 | Webhook触发→处理→推送→等待 |

### 1.3 OpenClaw中的工具编排

在OpenClaw中，工具编排是这样工作的：

- **System Prompt** 定义可用工具和行为规则（昨天学的！）
- **调度器（Scheduler）** 管理定时任务的执行
- **事件总线（Event Bus）** 接收和分发外部事件
- **路由器（Router）** 将消息发到正确通道（微信/Telegram等）
- **节点（Node）** 可连接手机等设备实现通知推送

> 💡 **关键洞察**：工具编排不是让LLM一次性决定所有步骤，而是让LLM在每个"决策点"做判断，由编排框架管理流程控制。


In [ ]:
# ===========================
# 🔧 中文字体配置（必须首先执行）
# ===========================
from matplotlib import font_manager
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("✅ 字体配置完成，使用字体:", font_name)


In [ ]:
# ===========================
# 📊 可视化：工具编排的五个层次
# ===========================

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

levels = [
    ("L0 单工具调用", 9.0, "#E3F2FD", "#1565C0", "用户→工具→回复"),
    ("L1 顺序链", 7.5, "#E8F5E9", "#2E7D32", "工具A→工具B→工具C"),
    ("L2 条件分支", 6.0, "#FFF3E0", "#E65100", "判断→路径A / 路径B"),
    ("L3 并行执行", 4.5, "#F3E5F5", "#6A1B9A", "同时运行多个工具"),
    ("L4 事件循环", 3.0, "#FFEBEE", "#B71C1C", "持续监听→响应→推送"),
]

for i, (title, y, bg_color, text_color, desc) in enumerate(levels):
    width = 7 + i * 0.3
    x_start = 5 - width / 2
    rect = FancyBboxPatch(
        (x_start, y - 0.6), width, 1.0,
        boxstyle="round,pad=0.1",
        facecolor=bg_color, edgecolor=text_color, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x_start + 0.3, y + 0.15, title,
            fontsize=13, fontweight='bold', color=text_color)
    ax.text(x_start + 0.3, y - 0.2, f"💡 {desc}",
            fontsize=10, color='#555555')

for i in range(len(levels) - 1):
    y_top = levels[i][1] - 0.7
    y_bot = levels[i + 1][1] + 0.5
    ax.annotate('', xy=(5, y_bot), xytext=(5, y_top),
                arrowprops=dict(arrowstyle='->', color='#666666', lw=1.5))
    ax.text(5.3, (y_top + y_bot) / 2, '复杂度↑', fontsize=9,
            color='#888888', va='center')

ax.text(5, 9.7, '工具编排的五个层次：从W5到Day2',
        fontsize=15, fontweight='bold', ha='center', color='#333333')

plt.tight_layout()
plt.show()
print("📈 图表完成：工具编排层次图")


---

## 2️⃣ 定时任务（Cron）与调度

### 2.1 什么是Cron？

**Cron** 是Unix/Linux系统中用于定时执行任务的调度器。在Agent系统中，我们用Cron来实现：

| 场景 | Cron表达式 | 说明 |
|------|-----------|------|
| 每天早上8点推送AI新闻 | `0 8 * * *` | 每日定时触发 |
| 每周一9点生成周报 | `0 9 * * 1` | 每周固定日触发 |
| 每小时检查新邮件 | `0 * * * *` | 小时级周期 |
| 每30分钟同步数据 | `*/30 * * * *` | 分钟级周期 |
| 工作日下班提醒 | `0 18 * * 1-5` | 仅工作日触发 |

### 2.2 Cron表达式语法

```
┌───────────── 分钟 (0-59)
│ ┌───────────── 小时 (0-23)
│ │ ┌───────────── 日 (1-31)
│ │ │ ┌───────────── 月 (1-12)
│ │ │ │ ┌───────────── 星期 (0-6, 0=周日)
│ │ │ │ │
* * * * *
```

| 符号 | 含义 | 例子 |
|------|------|------|
| `*` | 任意值 | 每分钟/每小时 |
| `,` | 列举 | `1,15` → 第1和第15分钟 |
| `-` | 范围 | `1-5` → 周一到周五 |
| `*/n` | 每隔n | `*/5` → 每5分钟 |

### 2.3 Agent系统中的Cron应用

```
Cron触发器
    │
    ▼
Agent处理器          System Prompt中定义：
┌──────────┐       "每天8点执行AI新闻摘要"
│ 1. 调用搜索 │       "使用web_search获取AI新闻"
│ 2. LLM总结 │       "调用tts朗读给用户听"
│ 3. 格式化  │
│ 4. 推送    │
└──────────┘
    │
    ├──▶ 微信推送
    ├──▶ Telegram推送
    └──▶ 邮件推送
```

> 💡 **在OpenClaw中**：Cron定时任务由Gateway调度器管理，触发后会唤醒对应的Agent处理流程，完成后通过配置的通道（如微信）推送结果。


In [ ]:
# ===========================
# 💻 模拟：Cron表达式解析器
# ===========================

def parse_cron(expression: str) -> dict:
    """解析Cron表达式，返回人类可读的描述"""
    parts = expression.strip().split()
    if len(parts) != 5:
        return {"error": "Cron表达式必须有5个字段"}
    
    minute, hour, day, month, weekday = parts
    weekday_names = ["周日", "周一", "周二", "周三", "周四", "周五", "周六"]
    
    def describe_field(value, field_name, names=None):
        if value == '*':
            return f"每{field_name}"
        elif value.startswith('*/'):
            interval = value[2:]
            return f"每{interval}个{field_name}"
        elif '-' in value:
            start, end = value.split('-')
            if names:
                return f"{names[int(start)]}到{names[int(end)]}"
            return f"第{start}到第{end}{field_name}"
        elif ',' in value:
            items = value.split(',')
            return f"{field_name}为{'、'.join(items)}"
        else:
            if names and value.isdigit():
                return names[int(value)]
            return f"第{value}{field_name}"
    
    return {
        "表达式": expression,
        "分钟": describe_field(minute, "分钟"),
        "小时": describe_field(hour, "小时"),
        "日": describe_field(day, "天"),
        "月": describe_field(month, "月"),
        "星期": describe_field(weekday, "星期", weekday_names),
    }

# 测试几个常见的Cron表达式
cron_examples = [
    ("0 8 * * *", "每日AI新闻推送"),
    ("0 9 * * 1", "每周一晨会摘要"),
    ("*/30 * * * *", "每30分钟数据同步"),
    ("0 18 * * 1-5", "工作日下班提醒"),
    ("0 0 1 * *", "每月1日报告生成"),
]

print("=" * 65)
print(f"{'Cron表达式':<16} {'任务名称':<18} {'解析结果'}")
print("=" * 65)

for expr, task_name in cron_examples:
    parsed = parse_cron(expr)
    desc = f'"{parsed["小时"]}" "{parsed["星期"]}'
    print(f"{expr:<16} {task_name:<18} {desc}")

print("=" * 65)
print("\n✅ Cron解析器演示完成")


In [ ]:
# ===========================
# 📊 可视化：24小时Cron任务时间线
# ===========================

fig, ax = plt.subplots(figsize=(14, 6))

tasks = [
    {"name": "AI新闻推送", "cron": "0 8 * * *", "hours": [8], "color": "#1976D2"},
    {"name": "数据同步", "cron": "*/6 * * * *", "hours": [0, 6, 12, 18], "color": "#388E3C"},
    {"name": "邮件检查", "cron": "0 */3 * * *", "hours": [3, 6, 9, 12, 15, 18, 21], "color": "#F57C00"},
    {"name": "健康提醒", "cron": "0 12,18 * * *", "hours": [12, 18], "color": "#7B1FA2"},
    {"name": "日报生成", "cron": "30 17 * * 1-5", "hours": [17.5], "color": "#D32F2F"},
    {"name": "周报汇总", "cron": "0 9 * * 1", "hours": [9], "color": "#00796B"},
]

hours = np.arange(0, 24)
ax.set_xlim(-0.5, 24)
ax.set_ylim(-0.5, len(tasks) + 0.5)

for h in hours:
    ax.axvline(x=h, color='#EEEEEE', linewidth=0.5, zorder=0)
    if h % 3 == 0:
        ax.text(h, len(tasks) + 0.2, f'{h:02d}:00', ha='center', fontsize=8, color='#666')

for i, task in enumerate(tasks):
    y = len(tasks) - 1 - i
    ax.barh(y, 24, height=0.6, color='#F5F5F5', edgecolor='#E0E0E0', zorder=1)
    for h in task["hours"]:
        ax.plot(h, y, 'o', color=task["color"], markersize=12, zorder=3)
        ax.plot(h, y, 'o', color='white', markersize=6, zorder=4)
    ax.text(-0.3, y, task["name"], ha='right', va='center',
            fontsize=10, fontweight='bold', color=task["color"])

ax.set_title('Agent定时任务24小时时间线', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('时间（小时）', fontsize=11)
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

legend_elements = [mpatches.Patch(facecolor=t["color"], label=f'{t["name"]} ({t["cron"]})')
                   for t in tasks]
ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=3, fontsize=8, frameon=False)

plt.tight_layout()
plt.show()
print("📈 图表完成：Cron任务时间线")


---

## 3️⃣ 跨平台消息路由

### 3.1 为什么需要消息路由？

现实中，你的消息来源和目标可能是多个平台的：

| 场景 | 来源 | 目标 | 处理方式 |
|------|------|------|----------|
| 客户咨询 | 微信 | Agent处理→微信回复 | 即时响应 |
| 监控告警 | Webhook | Agent分析→Telegram通知 | 事件驱动 |
| 定时报告 | Cron | Agent生成→邮件+微信 | 定时推送 |
| 团队协作 | Slack | Agent转发→飞书/钉钉 | 跨平台桥接 |

### 3.2 消息路由架构

```
                  ┌─────────┐
    微信 ─────────▶│         │
    Telegram ─────▶│  消息    │──▶ 意图识别 ──▶ 工具调用
    Slack ────────▶│  路由器  │                  │
    Webhook ──────▶│         │                  ▼
    API调用 ───────▶└─────────┘            ┌──────────┐
                                           │ LLM处理  │
                                           └────┬─────┘
                                                │
                  ┌─────────┐                   │
    微信 ◀────────│         │◀──────────────────┘
    Telegram ◀────│  响应    │
    邮件 ◀────────│  分发器  │
    飞书 ◀────────│         │
                  └─────────┘
```

### 3.3 消息路由的核心组件

| 组件 | 功能 | 类比 |
|------|------|------|
| **消息入口（Inbound）** | 接收各平台消息 | 快递收件箱 |
| **意图识别（Intent）** | 理解消息要做什么 | 分拣中心 |
| **处理引擎（Engine）** | 调用工具执行任务 | 工人 |
| **响应分发（Outbound）** | 发送结果到目标平台 | 快递发件箱 |
| **状态管理（State）** | 保存上下文和会话 | 记事本 |

### 3.4 路由策略

```
策略1：来源回传 — 消息从哪来回哪去
    微信来 → 处理 → 微信回

策略2：规则路由 — 根据内容选择通道
    紧急消息 → 微信（即时）
    报告邮件 → 邮件（正式）
    团队通知 → 飞书（协作）

策略3：多通道广播 — 同时推送到多个平台
    AI日报 → 微信 + Telegram + 邮件

策略4：智能选择 — LLM决定发到哪
    "帮我提醒Jason开会" → LLM判断Jason常用微信 → 微信推送
```

> 💡 **OpenClaw实战**：OpenClaw的 `openclaw-weixin` 插件就是消息路由的实现——微信消息进入 → Agent处理 → 结果通过微信或其他通道返回。


In [ ]:
# ===========================
# 💻 模拟：消息路由器实现
# ===========================

from dataclasses import dataclass
from enum import Enum

class Platform(Enum):
    WECHAT = "微信"
    TELEGRAM = "Telegram"
    EMAIL = "邮件"
    FEISHU = "飞书"
    WEBHOOK = "Webhook"

class Urgency(Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"
    CRITICAL = "紧急"

@dataclass
class Message:
    content: str
    source: Platform
    urgency: Urgency = Urgency.MEDIUM
    user_id: str = "jason"
    timestamp: str = "2026-07-14 14:30:00"

@dataclass
class RoutingDecision:
    message: Message
    intent: str
    target_platforms: list
    tools_needed: list
    processing_time_ms: int

class MessageRouter:
    def __init__(self):
        self.rules = {
            Urgency.CRITICAL: [Platform.WECHAT, Platform.TELEGRAM],
            Urgency.HIGH: [Platform.WECHAT],
            Urgency.MEDIUM: [Platform.WECHAT, Platform.EMAIL],
            Urgency.LOW: [Platform.EMAIL],
        }
        self.route_count = 0
        self.history = []
    
    def identify_intent(self, message):
        content = message.content
        if any(w in content for w in ["天气", "weather"]):
            return "查天气"
        elif any(w in content for w in ["新闻", "news"]):
            return "获取新闻"
        elif any(w in content for w in ["提醒", "remind"]):
            return "设置提醒"
        elif any(w in content for w in ["告警", "alert", "异常"]):
            return "处理告警"
        else:
            return "通用对话"
    
    def route(self, message):
        self.route_count += 1
        intent = self.identify_intent(message)
        targets = self.rules.get(message.urgency, [Platform.WECHAT])
        intent_tools = {
            "查天气": ["weather_api", "location_service"],
            "获取新闻": ["web_search", "summary_llm"],
            "设置提醒": ["scheduler", "notification"],
            "处理告警": ["analyzer", "notification", "escalation"],
            "通用对话": ["llm_chat"],
        }
        tools = intent_tools.get(intent, ["llm_chat"])
        decision = RoutingDecision(
            message=message, intent=intent,
            target_platforms=targets, tools_needed=tools,
            processing_time_ms=150 + self.route_count * 12,
        )
        self.history.append(decision)
        return decision

# ===== 模拟多条消息路由 =====
router = MessageRouter()

test_messages = [
    Message("今天北京天气怎么样？", Platform.WECHAT, Urgency.LOW),
    Message("帮我获取今天的AI新闻", Platform.WECHAT, Urgency.MEDIUM),
    Message("⚠️ 服务器CPU使用率超过95%！", Platform.WEBHOOK, Urgency.CRITICAL),
    Message("明天下午3点提醒我开会", Platform.TELEGRAM, Urgency.HIGH),
    Message("最近的AI论文有什么推荐的？", Platform.WECHAT, Urgency.LOW),
]

print("=" * 60)
print("🚀 消息路由器演示 — 5条消息的智能路由")
print("=" * 60)
print()

for msg in test_messages:
    d = router.route(msg)
    print(f"📨 消息: \"{d.message.content}\"")
    print(f"   来源: {d.message.source.value}")
    print(f"   紧急: {d.message.urgency.value}")
    print(f"   意图: {d.intent}")
    print(f"   工具: {', '.join(d.tools_needed)}")
    print(f"   推送: {', '.join(p.value for p in d.target_platforms)}")
    print(f"   耗时: {d.processing_time_ms}ms")
    print()

print(f"📊 总计处理 {router.route_count} 条消息")


In [ ]:
# ===========================
# 📊 可视化：多通道消息流量热力图
# ===========================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

platforms = ["微信", "Telegram", "邮件", "飞书", "Webhook"]
hours_labels = ["6-9时", "9-12时", "12-14时", "14-17时", "17-20时", "20-24时"]

source_data = np.array([
    [12, 45, 15, 38, 22, 8],
    [3, 15, 8, 20, 12, 25],
    [5, 25, 30, 20, 15, 5],
    [2, 30, 10, 35, 18, 3],
    [1, 5, 2, 8, 3, 1],
])

im1 = ax1.imshow(source_data, cmap='YlOrRd', aspect='auto')
ax1.set_xticks(range(len(hours_labels)))
ax1.set_xticklabels(hours_labels, fontsize=9)
ax1.set_yticks(range(len(platforms)))
ax1.set_yticklabels(platforms, fontsize=10)
ax1.set_title('消息来源 × 时段 流量', fontsize=12, fontweight='bold')

for i in range(len(platforms)):
    for j in range(len(hours_labels)):
        color = 'white' if source_data[i, j] > 25 else 'black'
        ax1.text(j, i, str(source_data[i, j]), ha='center', va='center',
                color=color, fontsize=9, fontweight='bold')

plt.colorbar(im1, ax=ax1, shrink=0.8, label='消息量')

message_types = ["即时对话", "定时报告", "告警通知", "文件分享", "任务协作"]
sizes = [35, 20, 15, 12, 18]
colors_pie = ["#2196F3", "#4CAF50", "#F44336", "#FF9800", "#9C27B0"]
explode = (0.05, 0, 0.05, 0, 0)

ax2.pie(sizes, explode=explode, labels=message_types, autopct='%1.1f%%',
        colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('消息类型分布', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print("📈 图表完成：消息路由可视化")


---

## 4️⃣ 工作流设计模式

### 4.1 什么是工作流？

**工作流（Workflow）** 是一系列有组织的步骤，用于完成特定任务。在Agent系统中：

```
触发条件 → 数据获取 → LLM处理 → 结果格式化 → 推送通知
```

### 4.2 常见工作流模式

#### 模式1：线性管道（Pipeline）
```
输入 → 步骤A → 步骤B → 步骤C → 输出
     (搜索)  (总结)  (翻译)
```
**适用场景**：步骤之间有明确依赖，每步依赖上一步结果

#### 模式2：扇出-汇聚（Fan-out/Fan-in）
```
         → 工具A ──┐
输入 ──→  → 工具B ──┼──→ 合并 → 输出
         → 工具C ──┘
```
**适用场景**：多个独立任务可以并行执行

#### 模式3：条件分支（Conditional）
```
          ┌─ 条件满足 → 路径A → 工具X
输入 → 判断
          └─ 条件不满足 → 路径B → 工具Y
```
**适用场景**：需要根据中间结果做决策

#### 模式4：事件循环（Event Loop）
```
触发 → 处理 → 输出 → 等待 → 触发 → 处理 → ...
```
**适用场景**：需要持续运行的长期任务

### 4.3 工作流状态管理

| 状态 | 含义 | 示例 |
|------|------|------|
| **IDLE** | 等待触发 | Cron任务等待下一个时间点 |
| **RUNNING** | 正在执行 | Agent正在搜索和总结新闻 |
| **WAITING** | 等待外部响应 | 等待API返回数据 |
| **SUCCESS** | 执行成功 | 新闻已推送到微信 |
| **FAILED** | 执行失败 | API超时，需要重试 |
| **PAUSED** | 暂停等待 | 等待用户确认 |

> 💡 **设计原则**：好的工作流应该有**幂等性**（重复执行不产生副作用）、**可观测性**（能看到每步状态）、和**容错性**（失败后能恢复）。


In [ ]:
# ===========================
# 💻 模拟：工作流状态机实现
# ===========================

from enum import Enum
from datetime import datetime
import time

class WorkflowState(Enum):
    IDLE = "空闲"
    RUNNING = "执行中"
    WAITING = "等待中"
    SUCCESS = "成功"
    FAILED = "失败"
    PAUSED = "暂停"

class WorkflowStep:
    def __init__(self, name, tool_name, duration_ms=500):
        self.name = name
        self.tool_name = tool_name
        self.duration_ms = duration_ms
        self.status = None
        self.result = None
    
    def execute(self, context):
        self.status = "running"
        print(f"  ⚙️  [{self.name}] 调用工具: {self.tool_name}")
        time.sleep(self.duration_ms / 1000)
        self.status = "success"
        self.result = f"{self.tool_name}返回结果"
        print(f"  ✅ [{self.name}] 完成 ({self.duration_ms}ms)")
        return {"tool": self.tool_name, "result": self.result}

class Workflow:
    def __init__(self, name):
        self.name = name
        self.state = WorkflowState.IDLE
        self.steps = []
        self.context = {}
        self.state_history = [(datetime.now(), WorkflowState.IDLE)]
    
    def add_step(self, step):
        self.steps.append(step)
        return self
    
    def run(self, input_context=None):
        self.state = WorkflowState.RUNNING
        self.context = input_context or {}
        self.state_history.append((datetime.now(), self.state))
        print(f"\n🔄 工作流 [{self.name}] 开始执行")
        print(f"   状态: {self.state.value}")
        print(f"   步骤数: {len(self.steps)}")
        print()
        
        for step in self.steps:
            try:
                step_result = step.execute(self.context)
                self.context[f"step_{step.name}"] = step_result
            except Exception as e:
                self.state = WorkflowState.FAILED
                print(f"  ❌ [{step.name}] 失败: {e}")
                self.state_history.append((datetime.now(), self.state))
                return False
        
        self.state = WorkflowState.SUCCESS
        self.state_history.append((datetime.now(), self.state))
        print(f"\n🎉 工作流 [{self.name}] 执行完成!")
        print(f"   最终状态: {self.state.value}")
        transitions = ' → '.join(s.value for _, s in self.state_history)
        print(f"   状态变更: {transitions}")
        return True

# ===== 示例：每日AI新闻推送工作流 =====
workflow = Workflow("每日AI新闻推送")
workflow.add_step(WorkflowStep("搜索新闻", "web_search", 300))
workflow.add_step(WorkflowStep("提取内容", "content_extractor", 200))
workflow.add_step(WorkflowStep("LLM总结", "summary_llm", 800))
workflow.add_step(WorkflowStep("格式化", "formatter", 150))
workflow.add_step(WorkflowStep("推送微信", "wechat_push", 400))

workflow.run({"date": "2026-07-14", "user": "Jason"})


In [ ]:
# ===========================
# 📊 可视化：四种工作流模式对比
# ===========================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('四种工作流设计模式', fontsize=15, fontweight='bold', y=0.98)

# ---- 模式1: 线性管道 ----
ax = axes[0, 0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('模式1: 线性管道 (Pipeline)', fontsize=11, fontweight='bold')
steps = [("输入", 1), ("搜索", 3), ("总结", 5), ("推送", 7), ("输出", 9)]
for name, x in steps:
    rect = FancyBboxPatch((x-0.6, 2.5), 1.2, 1, boxstyle="round,pad=0.1",
                          facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 3, name, ha='center', va='center', fontsize=9, fontweight='bold')
for i in range(len(steps)-1):
    ax.annotate('', xy=(steps[i+1][1]-0.7, 3), xytext=(steps[i][1]+0.7, 3),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.text(5, 1.2, '步骤间有依赖关系', ha='center', fontsize=9, color='#666')

# ---- 模式2: 扇出-汇聚 ----
ax = axes[0, 1]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('模式2: 扇出-汇聚 (Fan-out/Fan-in)', fontsize=11, fontweight='bold')
rect = FancyBboxPatch((0.2, 2.5), 1.2, 1, boxstyle="round,pad=0.1",
                      facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(rect)
ax.text(0.8, 3, "输入", ha='center', va='center', fontsize=9, fontweight='bold')
for i, (name, y) in enumerate([("工具A", 4.5), ("工具B", 3), ("工具C", 1.5)]):
    rect = FancyBboxPatch((3.5, y-0.4), 1.2, 0.8, boxstyle="round,pad=0.1",
                          facecolor='#F3E5F5', edgecolor='#7B1FA2', linewidth=2)
    ax.add_patch(rect)
    ax.text(4.1, y, name, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(3.4, y), xytext=(1.5, 3),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5))
rect = FancyBboxPatch((6.5, 2.5), 1.2, 1, boxstyle="round,pad=0.1",
                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect)
ax.text(7.1, 3, "合并", ha='center', va='center', fontsize=9, fontweight='bold')
for y in [4.5, 3, 1.5]:
    ax.annotate('', xy=(6.4, 3), xytext=(4.8, y),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5))
ax.text(5, 0.5, '并行执行独立任务', ha='center', fontsize=9, color='#666')

# ---- 模式3: 条件分支 ----
ax = axes[1, 0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('模式3: 条件分支 (Conditional)', fontsize=11, fontweight='bold')
rect = FancyBboxPatch((0.2, 2.5), 1.2, 1, boxstyle="round,pad=0.1",
                      facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(rect)
ax.text(0.8, 3, "输入", ha='center', va='center', fontsize=9, fontweight='bold')
diamond = plt.Polygon([(3.5, 3.8), (4.5, 3), (3.5, 2.2), (2.5, 3)],
                      facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(diamond)
ax.text(3.5, 3, "判断", ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(2.4, 3), xytext=(1.5, 3),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
rect = FancyBboxPatch((5.5, 4.2), 1.4, 0.8, boxstyle="round,pad=0.1",
                      facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2)
ax.add_patch(rect)
ax.text(6.2, 4.6, "高优先级", ha='center', va='center', fontsize=9, color='#D32F2F')
ax.annotate('', xy=(5.4, 4.6), xytext=(4.6, 3.4),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=1.5))
ax.text(4.3, 4.2, '是', fontsize=8, color='#D32F2F')
rect = FancyBboxPatch((5.5, 1.2), 1.4, 0.8, boxstyle="round,pad=0.1",
                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect)
ax.text(6.2, 1.6, "低优先级", ha='center', va='center', fontsize=9, color='#2E7D32')
ax.annotate('', xy=(5.4, 1.6), xytext=(4.6, 2.6),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5))
ax.text(4.3, 1.6, '否', fontsize=8, color='#2E7D32')
ax.text(5, 0.4, '根据条件选择路径', ha='center', fontsize=9, color='#666')

# ---- 模式4: 事件循环 ----
ax = axes[1, 1]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('模式4: 事件循环 (Event Loop)', fontsize=11, fontweight='bold')
loop_items = [(2, 4.5, "触发"), (5, 4.5, "处理"), (8, 4.5, "输出"),
              (8, 1.8, "等待"), (5, 1.8, "监听"), (2, 1.8, "就绪")]
for x, y, name in loop_items:
    rect = FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle="round,pad=0.1",
                          facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=9,
            fontweight='bold', color='#2E7D32')
arrow_pairs = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)]
for start, end in arrow_pairs:
    sx, sy = loop_items[start]; ex, ey = loop_items[end]
    ax.annotate('', xy=(ex, ey+0.45), xytext=(sx, sy-0.45),
                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2,
                               connectionstyle='arc3,rad=0.15'))
ax.text(5, 0.5, '持续运行，循环处理', ha='center', fontsize=9, color='#666')

plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.show()
print("📈 图表完成：工作流模式对比")


---

## 5️⃣ 事件驱动架构

### 5.1 什么是事件驱动？

**事件驱动架构（Event-Driven Architecture）** 是一种系统设计模式，系统的行为由"事件"触发，而不是由用户主动调用。

在Agent系统中，事件可以是：

| 事件类型 | 来源 | 例子 |
|---------|------|------|
| **消息事件** | 用户发送消息 | 微信收到"今天天气怎样" |
| **定时事件** | Cron触发 | 每天8点执行新闻摘要 |
| **Webhook事件** | 外部系统通知 | GitHub提交了新代码 |
| **状态变更事件** | 内部系统变化 | 订单状态变为"已发货" |
| **传感器事件** | IoT设备 | 温度超过阈值 |

### 5.2 事件驱动 vs 请求-响应

```
请求-响应模式（传统）：
  用户 ──请求──▶ Agent ──响应──▶ 用户
  （每次都要用户主动触发）

事件驱动模式（自动化）：
  事件A ──▶ Agent处理 ──▶ 自动推送
  事件B ──▶ Agent处理 ──▶ 自动推送
  事件C ──▶ Agent处理 ──▶ 自动推送
  （系统自动响应，不需要用户主动操作）
```

### 5.3 事件处理流水线

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│ 事件接收  │───▶│ 事件过滤  │───▶│ 意图识别  │───▶│ 任务执行  │───▶│ 结果分发  │
│ (Ingress)│    │ (Filter) │    │ (Intent) │    │ (Execute)│    │ (Dispatch)│
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### 5.4 事件队列与背压策略

| 策略 | 说明 | 适用场景 |
|------|------|----------|
| **FIFO（先进先出）** | 按到达顺序处理 | 一般消息 |
| **优先级队列** | 紧急事件优先 | 告警 + 普通消息混合 |
| **丢弃最旧** | 队列满时丢弃旧事件 | 实时性要求高 |
| **限流（Rate Limit）** | 控制处理速率 | 防止过载 |
| **死信队列** | 处理失败的事件存入单独队列 | 需要重试 |

> 💡 **实战思考**：OpenClaw中，微信消息到达就是一个"事件"，Gateway接收后触发Agent处理——这就是典型的事件驱动。Cron定时器也是一种特殊的事件源。


In [ ]:
# ===========================
# 💻 模拟：事件驱动Agent系统
# ===========================

from collections import deque
from datetime import datetime

class Event:
    def __init__(self, event_type, source, data, priority=1):
        self.event_type = event_type
        self.source = source
        self.data = data
        self.priority = priority  # 1=低, 2=中, 3=高
        self.created_at = datetime.now()
        self.processed = False

class EventBus:
    def __init__(self, max_queue_size=20):
        self.queue = deque()
        self.max_size = max_queue_size
        self.processed_count = 0
        self.dropped_count = 0
        self.handlers = {}
    
    def register_handler(self, event_type, handler_func):
        self.handlers[event_type] = handler_func
    
    def emit(self, event):
        if len(self.queue) >= self.max_size:
            self.dropped_count += 1
            print(f"  ⚠️ 队列已满，丢弃事件: {event.event_type}")
            return False
        self.queue.append(event)
        p_label = '🔴高' if event.priority==3 else '🟡中' if event.priority==2 else '🟢低'
        print(f"  📨 事件入队: [{event.event_type}] 来源={event.source} 优先级={p_label}")
        return True
    
    def process_next(self):
        if not self.queue:
            print("  💤 队列为空，无事件处理")
            return
        sorted_queue = sorted(self.queue, key=lambda e: e.priority, reverse=True)
        event = sorted_queue[0]
        self.queue.remove(event)
        print(f"\n  ⚡ 处理事件: [{event.event_type}] 数据={event.data}")
        handler = self.handlers.get(event.event_type)
        if handler:
            handler(event)
        else:
            print(f"    🔧 默认处理: 记录日志 → LLM分析 → 生成回复")
        event.processed = True
        self.processed_count += 1
    
    def stats(self):
        print(f"\n📊 事件总线统计:")
        print(f"   已处理: {self.processed_count}")
        print(f"   队列中: {len(self.queue)}")
        print(f"   已丢弃: {self.dropped_count}")

# ===== 事件处理器 =====
def handle_message(event):
    print(f"    📩 消息处理: Agent接收 → 意图识别 → 工具调用 → 回复")

def handle_alert(event):
    print(f"    🚨 告警处理: 严重性分析 → 通知升级 → 推送")

def handle_webhook(event):
    print(f"    🔗 Webhook处理: 验证签名 → 提取数据 → 触发工作流")

def handle_timer(event):
    print(f"    ⏰ 定时处理: 唤醒Agent → 执行任务 → 推送结果")

# ===== 模拟运行 =====
bus = EventBus(max_queue_size=15)
bus.register_handler("message", handle_message)
bus.register_handler("alert", handle_alert)
bus.register_handler("webhook", handle_webhook)
bus.register_handler("timer", handle_timer)

print("=" * 55)
print("🎭 事件驱动Agent系统演示")
print("=" * 55)

print("\n📥 接收事件...")
for e in [
    Event("timer", "cron", "每日新闻推送任务", 2),
    Event("message", "微信", "今天天气怎么样", 2),
    Event("message", "Telegram", "帮我翻译这段文字", 1),
    Event("alert", "监控", "CPU使用率超过95%", 3),
    Event("webhook", "GitHub", "新PR: fix-bug-123", 1),
    Event("message", "微信", "明天的会议安排", 2),
    Event("alert", "监控", "内存使用异常", 3),
]:
    bus.emit(e)

print("\n🔄 开始批量处理...")
for _ in range(7):
    bus.process_next()

bus.stats()


In [ ]:
# ===========================
# 📊 可视化：事件处理流水线性能
# ===========================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 左图：事件处理时间分布
event_types = ["消息", "告警", "Webhook", "定时", "邮件"]
avg_times = [320, 180, 250, 450, 380]
colors_time = ["#2196F3", "#F44336", "#FF9800", "#4CAF50", "#9C27B0"]

bars = ax1.barh(event_types, avg_times, color=colors_time, height=0.6, edgecolor='white')
for bar, time_val in zip(bars, avg_times):
    ax1.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'{time_val}ms', va='center', fontsize=10, fontweight='bold')
ax1.set_xlabel('平均处理时间 (ms)', fontsize=10)
ax1.set_title('各类型事件平均处理时间', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 600)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.axvline(x=500, color='#D32F2F', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.text(505, len(event_types)-0.5, 'SLA: 500ms', fontsize=9, color='#D32F2F')

# 右图：24小时事件量
hours = np.arange(0, 24)
message_volume = 5 + 20*np.exp(-0.5*((hours-10)/3)**2) + 15*np.exp(-0.5*((hours-15)/2)**2)
alert_volume = 2 + 8*np.exp(-0.5*((hours-14)/2)**2)
webhook_volume = 3 + 5*np.exp(-0.5*((hours-16)/3)**2)
timer_volume = np.zeros(24)
timer_volume[8] = 10; timer_volume[9] = 8; timer_volume[17] = 12; timer_volume[18] = 10

ax2.stackplot(hours, message_volume, alert_volume, webhook_volume, timer_volume,
              labels=['消息', '告警', 'Webhook', '定时任务'],
              colors=['#2196F3', '#F44336', '#FF9800', '#4CAF50'], alpha=0.8)
ax2.set_xlabel('时间（小时）', fontsize=10)
ax2.set_ylabel('事件量', fontsize=10)
ax2.set_title('24小时事件量分布', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 23)
ax2.legend(loc='upper right', fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print("📈 图表完成：事件处理流水线性能")


---

## 6️⃣ Webhook与API集成

### 6.1 什么是Webhook？

**Webhook** 是一种"反向API"——不是你去调用别人，而是别人在事件发生时主动通知你。

```
普通API调用（Pull模式）：
  Agent ──定时请求──▶ 外部系统 ──返回数据──▶ Agent
  （需要Agent主动轮询）

Webhook（Push模式）：
  外部系统 ──事件触发──▶ Agent（Webhook URL）
  （外部系统主动推送）
```

### 6.2 Webhook工作原理

```
1. Agent注册Webhook URL
   POST https://your-agent.com/webhook/github
   
2. GitHub配置Webhook
   当有新PR时，发送POST请求到上面的URL
   
3. Agent收到Webhook
   POST body: {"event": "pull_request", "action": "opened", ...}
   
4. Agent处理
   解析事件 → LLM分析PR内容 → 生成摘要 → 推送到微信
```

### 6.3 常见Webhook集成场景

| 外部系统 | 触发事件 | Agent处理 | 输出 |
|---------|---------|----------|------|
| **GitHub/GitLab** | 新PR、新Issue | 分析代码变更 | 推送摘要到群聊 |
| **Stripe** | 支付成功/失败 | 更新订单状态 | 通知相关人员 |
| **Sentry** | 错误上报 | 分析错误原因 | 创建修复任务 |
| **Jenkins/CI** | 构建成功/失败 | 分析日志 | 通知开发团队 |
| **Shopify** | 新订单 | 处理订单 | 通知仓库发货 |
| **Slack** | 特定频道消息 | 监控关键词 | 跨平台转发 |

### 6.4 安全考虑

| 安全措施 | 说明 | 重要性 |
|---------|------|--------|
| **签名验证** | 验证Webhook来源真实性 | ⭐⭐⭐ 必须 |
| **HTTPS** | 加密传输数据 | ⭐⭐⭐ 必须 |
| **速率限制** | 防止恶意请求淹没系统 | ⭐⭐⭐ 必须 |
| **输入验证** | 验证和清洗传入数据 | ⭐⭐⭐ 必须 |
| **最小权限** | API Token只给必要权限 | ⭐⭐⭐ 必须 |

> 💡 **OpenClaw实战**：OpenClaw的节点系统可以接收Webhook，然后触发Agent处理。微信消息也可以看作是一种特殊的Webhook——消息到达就是事件触发。


In [ ]:
# ===========================
# 💻 模拟：Webhook处理器与API集成
# ===========================

import hashlib
import hmac
import json

class WebhookHandler:
    def __init__(self, secret="webhook_secret_key"):
        self.secret = secret.encode()
        self.handlers = {}
        self.request_log = []
    
    def register(self, event_type, handler):
        self.handlers[event_type] = handler
        print(f"✅ 注册处理器: {event_type}")
    
    def verify_signature(self, payload, signature):
        expected = hmac.new(self.secret, payload, hashlib.sha256).hexdigest()
        return hmac.compare_digest(expected, signature)
    
    def handle_request(self, method, path, body, signature=None):
        request_id = len(self.request_log) + 1
        if signature:
            payload_bytes = json.dumps(body).encode()
            if not self.verify_signature(payload_bytes, signature):
                print(f"\n🚫 请求 #{request_id}: 签名验证失败！")
                self.request_log.append({"id": request_id, "status": "rejected"})
                return {"error": "invalid_signature"}
        
        print(f"\n📨 请求 #{request_id}: {method} {path}")
        print(f"   事件: {body.get('event', 'unknown')}")
        
        event_type = body.get("event")
        handler = self.handlers.get(event_type)
        if handler:
            print(f"   → 调用处理器: {event_type}")
            result = handler(body)
            print(f"   ← 处理完成: {result.get('status')}")
        else:
            print(f"   ⚠️ 无匹配处理器，返回404")
            result = {"status": "not_found"}
        
        self.request_log.append({"id": request_id, "event": event_type, "status": result.get("status")})
        return result

# ===== Webhook处理器 =====
def handle_github_pr(body):
    pr = body.get("data", {})
    print(f"   🔍 分析PR: {pr.get('title', 'N/A')} by {pr.get('author', 'N/A')}")
    print(f"   📝 LLM分析代码变更...")
    return {"status": "processed", "action": "pr_summary_sent"}

def handle_stripe_payment(body):
    payment = body.get("data", {})
    amount = payment.get("amount", 0) / 100
    print(f"   💰 支付: ¥{amount:.2f} - {payment.get('status', 'unknown')}")
    print(f"   📋 更新订单状态...")
    return {"status": "processed", "action": "order_updated"}

def handle_sentry_error(body):
    error = body.get("data", {})
    print(f"   🐛 错误: {error.get('message', 'N/A')}")
    print(f"   🔎 分析堆栈... 创建修复任务...")
    return {"status": "processed", "action": "bug_task_created"}

# ===== 模拟运行 =====
handler = WebhookHandler()
handler.register("pull_request", handle_github_pr)
handler.register("payment.success", handle_stripe_payment)
handler.register("error.reported", handle_sentry_error)

print("=" * 55)
print("🔗 Webhook处理器演示 — 外部系统事件集成")
print("=" * 55)

for req in [
    {"method": "POST", "path": "/webhook/github", "body": {
        "event": "pull_request", "data": {
            "title": "feat: add streaming response support",
            "author": "developer-001", "files_changed": 12}}},
    {"method": "POST", "path": "/webhook/stripe", "body": {
        "event": "payment.success", "data": {
            "amount": 29900, "currency": "CNY", "status": "succeeded", "customer": "Jason"}}},
    {"method": "POST", "path": "/webhook/sentry", "body": {
        "event": "error.reported", "data": {
            "message": "TypeError: Cannot read property", "level": "error", "occurrences": 5}}},
]:
    handler.handle_request(req["method"], req["path"], req["body"])

print("\n📋 Webhook请求日志:")
for log in handler.request_log:
    icon = "✅" if log["status"] == "processed" else "❌"
    print(f"   {icon} #{log['id']} {log.get('event','?')} → {log['status']}")


In [ ]:
# ===========================
# 📊 可视化：API集成生态图
# ===========================

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

center_x, center_y = 5, 5
circle = plt.Circle((center_x, center_y), 1.2, facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=3)
ax.add_patch(circle)
ax.text(center_x, center_y + 0.3, "🤖 Agent", ha='center', va='center',
        fontsize=14, fontweight='bold', color='#1565C0')
ax.text(center_x, center_y - 0.3, "工具编排引擎", ha='center', va='center',
        fontsize=10, color='#1565C0')

external = [
    (5, 9, "GitHub", "#24292e", "PR/Issue事件"),
    (1.5, 7.5, "Stripe", "#635BFF", "支付通知"),
    (1.5, 2.5, "Sentry", "#F23D3E", "错误监控"),
    (5, 0.8, "Jenkins", "#D33833", "CI/CD"),
    (8.5, 2.5, "Shopify", "#96BF48", "电商订单"),
    (8.5, 7.5, "Slack", "#4A154B", "消息事件"),
]

for x, y, name, color, desc in external:
    rect = FancyBboxPatch((x-0.8, y-0.4), 1.6, 0.8, boxstyle="round,pad=0.1",
                          facecolor='white', edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y+0.15, name, ha='center', va='center', fontsize=11, fontweight='bold', color=color)
    ax.text(x, y-0.2, desc, ha='center', va='center', fontsize=8, color='#666')
    dx, dy = center_x - x, center_y - y
    dist = (dx**2 + dy**2) ** 0.5
    ax.annotate('', xy=(center_x - dx/dist*1.3, center_y - dy/dist*1.3),
                xytext=(x + dx/dist, y + dy/dist*0.5),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.6))

ax.set_title('Agent Webhook/API集成生态', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print("📈 图表完成：API集成生态图")


---

## 🏗️ 综合案例：每日AI新闻推送工作流

### 案例背景

Jason每天早上8点想收到一份AI领域新闻摘要，包含：
1. 最重要的3-5条AI新闻
2. 每条新闻的一句话总结
3. 可能的投资/产品机会提示
4. 通过微信推送，可以手机朗读收听

### 完整工作流设计

```
┌──────────────────────────────────────────────────────────┐
│                每日AI新闻推送工作流                        │
├──────────────────────────────────────────────────────────┤
│  ┌──────┐                                                │
│  │ Cron │ 每天早上 07:55 触发                               │
│  └──┬───┘                                                │
│     ▼                                                   │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐                  │
│  │ 搜索新闻  │─▶│ 提取正文  │─▶│ LLM总结   │                  │
│  │ web_search│ │ extractor│ │ summary  │                  │
│  └──────────┘ └──────────┘ └────┬─────┘                  │
│                                    │                      │
│  ┌──────────┐ ┌──────────┐        ▼                      │
│  │ TTS朗读   │◀─┤ 格式化   │◀──┌──────────┐              │
│  │ 语音生成  │ │ formatter│   │ 机会识别  │              │
│  └──────────┘ └────┬─────┘   │ analyzer │              │
│                    │         └──────────┘              │
│                    ▼                                      │
│  ┌──────────┐ ┌──────────┐                               │
│  │ 微信推送   │◀─┤ 消息路由  │                               │
│  └──────────┘ └──────────┘                               │
└──────────────────────────────────────────────────────────┘
```

### 技术要点

| 环节 | 技术 | 说明 |
|------|------|------|
| 触发 | Cron `55 7 * * *` | 每天7:55触发，确保8点前完成 |
| 搜索 | web_search 工具 | 搜索最新AI新闻 |
| 提取 | content_extractor | 提取网页正文 |
| 总结 | LLM + System Prompt | 用Day1学的提示词工程生成摘要 |
| 机会识别 | LLM + 领域知识库 | 识别投资/产品机会 |
| 格式化 | formatter | 生成适合朗读的纯文本 |
| 推送 | openclaw-weixin | 微信消息推送 |
| 朗读 | TTS | 手机朗读功能收听 |

> 💡 **这就是Day2的完整闭环**：Cron触发 → 事件驱动 → 工具编排 → 多步处理 → 跨平台推送。把Day1的System Prompt工程也串联进来了！


In [ ]:
# ===========================
# 💻 综合：每日AI新闻推送工作流完整实现
# ===========================

import time
from datetime import datetime

class NewsWorkflow:
    def __init__(self):
        self.name = "每日AI新闻推送"
        self.steps_log = []
        self.news_data = []
    
    def log_step(self, step_name, status, detail=""):
        self.steps_log.append({"step": step_name, "status": status,
                               "time": datetime.now().strftime("%H:%M:%S"), "detail": detail})
        icon = "✅" if status == "success" else "⏳" if status == "running" else "❌"
        print(f"  {icon} [{step_name}] {detail}")
    
    def step_1_search(self):
        self.log_step("1.搜索新闻", "running")
        time.sleep(0.3)
        self.news_data = [
            {"title": "OpenAI发布GPT-5 Turbo", "source": "TechCrunch", "relevance": 0.95},
            {"title": "Google Gemini 3.0开源发布", "source": "The Verge", "relevance": 0.90},
            {"title": "Anthropic完成$50亿融资", "source": "Bloomberg", "relevance": 0.85},
            {"title": "Meta推出开源多模态模型Llama 5", "source": "Reuters", "relevance": 0.80},
            {"title": "Apple Intelligence 2.0发布", "source": "9to5Mac", "relevance": 0.70},
        ]
        self.log_step("1.搜索新闻", "success", f"找到 {len(self.news_data)} 条相关新闻")
    
    def step_2_extract(self):
        self.log_step("2.提取正文", "running")
        time.sleep(0.2)
        for news in self.news_data:
            news["content"] = f"{news['title']}的详细内容...（模拟）"
        self.log_step("2.提取正文", "success", f"提取了 {len(self.news_data)} 篇正文")
    
    def step_3_summarize(self):
        self.log_step("3.LLM总结", "running")
        time.sleep(0.5)
        summaries = [
            "OpenAI发布GPT-5 Turbo，推理速度提升3倍，成本降低60%",
            "Google Gemini 3.0正式开源，支持原生多模态推理",
            "Anthropic完成$50亿F轮融资，估值达$600亿",
            "Meta开源Llama 5，参数量达4000亿，多模态性能SOTA",
            "Apple Intelligence 2.0深度整合设备端AI，隐私优先",
        ]
        for news, summary in zip(self.news_data, summaries):
            news["summary"] = summary
        self.log_step("3.LLM总结", "success", "生成了5条新闻摘要")
    
    def step_4_analyze(self):
        self.log_step("4.机会识别", "running")
        time.sleep(0.3)
        opportunities = [
            {"type": "投资", "desc": "Anthropic估值快速上升，关注AI安全赛道"},
            {"type": "产品", "desc": "设备端AI趋势明显，关注端侧部署方案"},
        ]
        self.log_step("4.机会识别", "success", f"识别了 {len(opportunities)} 个机会")
        return opportunities
    
    def step_5_format(self, opportunities):
        self.log_step("5.格式化", "running")
        time.sleep(0.15)
        lines = [f"📰 AI日报 | {datetime.now().strftime('%Y-%m-%d')}"]
        lines.append("=" * 40)
        for i, news in enumerate(self.news_data, 1):
            lines.append(f"\n{i}. {news['summary']}")
            lines.append(f"   来源: {news['source']}")
        lines.append("\n💡 机会提示:")
        for opp in opportunities:
            lines.append(f"  • [{opp['type']}] {opp['desc']}")
        self.formatted_output = "\n".join(lines)
        self.log_step("5.格式化", "success", f"生成 {len(self.formatted_output)} 字符")
    
    def step_6_push(self):
        self.log_step("6.推送通知", "running")
        time.sleep(0.2)
        print(f"\n  📱 推送内容预览:")
        print("  " + "-" * 38)
        for line in self.formatted_output[:200].split("\n")[:8]:
            print(f"  {line}")
        print("  " + "-" * 38)
        self.log_step("6.推送通知", "success", "已推送到微信 (Jason)")
    
    def run(self):
        start = time.time()
        print("=" * 50)
        print(f"🔄 启动工作流: {self.name}")
        print(f"   时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 50)
        
        self.step_1_search()
        self.step_2_extract()
        self.step_3_summarize()
        opportunities = self.step_4_analyze()
        self.step_5_format(opportunities)
        self.step_6_push()
        
        elapsed = (time.time() - start) * 1000
        print(f"\n🎉 工作流完成！总耗时: {elapsed:.0f}ms")
        print(f"\n📊 执行摘要:")
        for log in self.steps_log:
            icon = "✅" if log["status"] == "success" else "❌"
            print(f"   {icon} {log['time']} {log['step']} - {log['detail']}")

workflow = NewsWorkflow()
workflow.run()


In [ ]:
# ===========================
# 📊 可视化：完整工具编排系统架构
# ===========================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14); ax.set_ylim(0, 10); ax.axis('off')
ax.text(7, 9.7, 'Agent工具编排完整系统架构', fontsize=15, fontweight='bold', ha='center', color='#333')

# 事件源层
ax.text(0.3, 9.1, '📡 事件源层', fontsize=11, fontweight='bold', color='#1565C0')
sources = [(1.5,8.3,"⏰ Cron\n定时器","#4CAF50"),(4,8.3,"💬 用户\n消息","#2196F3"),
           (6.5,8.3,"🔗 Webhook\n外部系统","#FF9800"),(9,8.3,"📱 设备\n事件","#9C27B0"),(11.5,8.3,"📧 邮件\nAPI","#607D8B")]
for x, y, label, color in sources:
    rect = FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle="round,pad=0.08",
                          facecolor='white', edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, color=color)
for x, _, _, _ in sources:
    ax.annotate('', xy=(7, 7.3), xytext=(x, 7.9),
                arrowprops=dict(arrowstyle='->', color='#999', lw=1))

# 事件总线
rect = FancyBboxPatch((1, 7.0), 12, 0.5, boxstyle="round,pad=0.05",
                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect)
ax.text(7, 7.25, '🔄 事件总线 (Event Bus) — 接收·过滤·路由', ha='center',
        va='center', fontsize=10, fontweight='bold', color='#2E7D32')
ax.annotate('', xy=(7, 6.4), xytext=(7, 6.95),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# Agent处理层
ax.text(0.3, 6.2, '🤖 处理层', fontsize=11, fontweight='bold', color='#1565C0')
agent_rect = FancyBboxPatch((1, 4.8), 12, 1.4, boxstyle="round,pad=0.1",
                            facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(agent_rect)
for x, y, label in [(2.5,5.9,"意图识别\n(LLM)"),(5,5.9,"工具选择\n(LLM)"),
                    (7.5,5.9,"参数填充\n(LLM)"),(10,5.9,"工具执行\n(External)"),(12,5.2,"状态\n管理")]:
    rect = FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle="round,pad=0.08",
                          facecolor='white', edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, color='#1976D2')
for i in range(3):
    ax.annotate('', xy=(agent_rect.get_x() + 1.4 + i*2.5 + 0.8, 5.9),
                xytext=(agent_rect.get_x() + 1.4 + i*2.5 + 2.5 - 0.8, 5.9),
                arrowprops=dict(arrowstyle='->', color='#1976D2', lw=1.2))

# 工具层
ax.text(0.3, 4.3, '🔧 工具层', fontsize=11, fontweight='bold', color='#E65100')
for x, y, label in [(2,3.6,"web_search\n网页搜索"),(4.5,3.6,"llm_chat\n对话模型"),
                     (7,3.6,"scheduler\n定时调度"),(9.5,3.6,"formatter\n格式化"),(12,3.6,"notifier\n通知推送")]:
    rect = FancyBboxPatch((x-0.7, y-0.35), 1.4, 0.7, boxstyle="round,pad=0.08",
                          facecolor='#FFF3E0', edgecolor='#F57C00', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, color='#F57C00')
ax.annotate('', xy=(7, 4.1), xytext=(7, 4.75),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

# 输出层
ax.text(0.3, 2.8, '📤 输出层', fontsize=11, fontweight='bold', color='#7B1FA2')
for x, y, label, color in [(2,2.2,"📱 微信","#07C160"),(4.5,2.2,"✈️ Telegram","#0088CC"),
                            (7,2.2,"📧 邮件","#D44638"),(9.5,2.2,"📁 飞书","#3370FF"),(12,2.2,"🔊 TTS语音","#555")]:
    rect = FancyBboxPatch((x-0.7, y-0.3), 1.4, 0.6, boxstyle="round,pad=0.08",
                          facecolor='#F3E5F5', edgecolor=color, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color=color)
ax.annotate('', xy=(7, 2.55), xytext=(7, 3.2),
            arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=2))

ax.text(7, 1.2, '事件源 → 事件总线 → Agent处理 → 工具执行 → 多平台输出',
        ha='center', fontsize=11, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F5F5F5', edgecolor='#DDD'))
ax.text(7, 0.7, '💡 System Prompt (Day1) 定义Agent行为规则，工具编排 (Day2) 管理完整流程',
        ha='center', fontsize=9, color='#888')

plt.tight_layout()
plt.show()
print("📈 图表完成：完整系统架构图")


---

## ✏️ 练习题

### 练习1：Cron表达式设计（基础）

为以下场景设计Cron表达式：

| 场景 | 你的答案 |
|------|----------|
| 每天下午6点推送日报 | ________ |
| 每周一上午10点周报 | ________ |
| 每工作日下午5点提醒 | ________ |
| 每15分钟检查一次 | ________ |
| 每月最后一天23:59 | ________ |

### 练习2：工作流设计（中级）

设计一个"**邮件智能助手**"工作流，包含：
1. 触发方式（用Cron还是Webhook？为什么？）
2. 需要哪些工具？
3. 画出流程图
4. 考虑错误处理

### 练习3：消息路由设计（中级）

设计一个消息路由规则，满足以下需求：
- **紧急告警**：同时发微信+短信
- **普通消息**：只发微信
- **定时报告**：发邮件+微信
- **团队通知**：发飞书+微信

用Python伪代码实现路由逻辑。

### 练习4：系统设计（高级）

设计一个完整的"**AI客服系统**"架构，需要考虑：
1. 多渠道接入（微信、网页、APP）
2. 智能路由（简单问题自动回答，复杂问题转人工）
3. 知识库检索（RAG，关联W4内容）
4. 会话管理（关联W5 Function Calling）
5. 日志和监控

画出架构图并说明每个组件的作用。


In [ ]:
# ===========================
# 💻 练习参考答案
# ===========================

print("=" * 55)
print("📝 练习题参考答案")
print("=" * 55)

# ===== 练习1 =====
print("\n📋 练习1：Cron表达式")
print("-" * 40)
for desc, answer in [
    ("每天下午6点推送日报", "0 18 * * *"),
    ("每周一上午10点周报", "0 10 * * 1"),
    ("每工作日下午5点提醒", "0 17 * * 1-5"),
    ("每15分钟检查一次", "*/15 * * * *"),
    ("每月最后一天23:59", "59 23 28-31 * *  # 注：精确实现需脚本判断"),
]:
    print(f"  {desc:<20} → {answer}")

# ===== 练习2 =====
print("\n📋 练习2：邮件智能助手工作流")
print("-" * 40)
print('')
print('  触发方式: Webhook（邮件服务器推送新邮件通知）')
print('  ')
print('  工具链:')
print('  1. email_receiver  - 接收邮件内容')
print('  2. intent_detector - LLM识别邮件意图')
print('  3. priority_scorer - 评估紧急程度')
print('  4. auto_responder  - 自动生成回复草稿')
print('  5. calendar_check  - 检查日历安排会议')
print('  6. notification    - 推送通知')
print('  ')
print('  错误处理:')
print('  - API超时 → 重试3次，间隔递增')
print('  - LLM失败 → 使用模板回复 + 标记人工处理')
print('  - 网络中断 → 存入离线队列，恢复后处理')
print('')

# ===== 练习3 =====
print("📋 练习3：消息路由规则实现")
print("-" * 40)

def route_message(message_type, urgency):
    routing_rules = {
        "alert": {"high": ["微信", "短信"], "medium": ["微信", "邮件"], "low": ["邮件"]},
        "message": {"high": ["微信"], "medium": ["微信"], "low": ["微信"]},
        "report": {"high": ["微信", "邮件"], "medium": ["邮件", "微信"], "low": ["邮件"]},
        "team": {"high": ["飞书", "微信"], "medium": ["飞书", "微信"], "low": ["飞书"]},
    }
    rules = routing_rules.get(message_type, routing_rules["message"])
    return rules.get(urgency, ["微信"])

for msg_type, urgency in [("alert","high"),("message","medium"),
                           ("report","medium"),("team","medium"),("alert","low")]:
    targets = route_message(msg_type, urgency)
    u = {"high":"高","medium":"中","low":"低"}[urgency]
    t = {"alert":"告警","message":"消息","report":"报告","team":"团队"}[msg_type]
    print(f"  {t}({u}) → 推送到: {', '.join(targets)}")


---

## 🧠 课后测试

### 单选题（每题5分，共50分）

**Q1.** Cron表达式 `0 */3 * * *` 表示？
- A. 每天凌晨3点执行
- B. 每3小时执行一次
- C. 每月第3天执行
- D. 每3分钟执行一次

**Q2.** 事件驱动架构中，"事件"可以来自以下哪些来源？
- A. 用户消息
- B. 定时触发器
- C. 外部Webhook
- D. 以上都是

**Q3.** 工具编排中"扇出-汇聚"模式的适用场景是？
- A. 步骤之间有强依赖
- B. 多个独立任务可以并行执行
- C. 只需要执行一个工具
- D. 根据条件走不同路径

**Q4.** Webhook和普通API调用的核心区别是？
- A. Webhook更安全
- B. Webhook是Push模式，API是Pull模式
- C. Webhook只能用POST方法
- D. 没有区别

**Q5.** 消息路由中的"策略路由"是指？
- A. 消息从哪来回哪去
- B. 根据内容/紧急程度选择不同通道
- C. 随机选择通道
- D. 只发到一个固定通道

### 判断题（每题5分，共25分）

**Q6.** System Prompt和工具编排是独立的，没有关联。（  ）

**Q7.** Cron定时任务是事件驱动架构中的一种事件源。（  ）

**Q8.** 工具编排中，LLM需要在开始时一次性决定所有步骤。（  ）

**Q9.** 死信队列用于存储处理失败的事件以便后续重试。（  ）

**Q10.** Webhook不需要验证签名，因为只接收不发送数据。（  ）

### 简答题（每题12.5分，共25分）

**Q11.** 请描述"事件循环"工作流模式，并举一个实际例子说明它适合什么场景。

**Q12.** 设计一个"会议室预约助手"的完整工作流，需要包含触发方式、工具链、输出通道和错误处理策略。


In [ ]:
# ===========================
# 📝 课后测试答案
# ===========================

print("=" * 55)
print("🧠 课后测试答案")
print("=" * 55)

print("\n📌 单选题答案:")
print("-" * 40)
for q, ans, explain in [
    ("Q1", "B", "0 */3 * * * 表示每3小时执行一次（*/n表示每隔n）"),
    ("Q2", "D", "事件可来自用户消息、定时器、Webhook等任何来源"),
    ("Q3", "B", "扇出-汇聚适合多个独立任务并行执行后合并结果"),
    ("Q4", "B", "Webhook是外部系统主动推送（Push），API是主动调用（Pull）"),
    ("Q5", "B", "策略路由根据消息内容/紧急程度选择不同通道"),
]:
    print(f"  {q}. {ans} — {explain}")

print("\n📌 判断题答案:")
print("-" * 40)
for q, ans, explain in [
    ("Q6", "❌ 错", "System Prompt定义Agent能力范围，工具编排管理执行流程"),
    ("Q7", "✅ 对", "Cron定时器就是定期产生'定时事件'的事件源"),
    ("Q8", "❌ 错", "LLM在每个决策点动态判断，不需要预知全部步骤"),
    ("Q9", "✅ 对", "死信队列（DLQ）专门存储失败事件，支持后续重试"),
    ("Q10", "❌ 错", "Webhook必须验证签名来确认请求确实来自可信来源"),
]:
    print(f"  {q}. {ans} — {explain}")

print("\n📌 简答题参考答案:")
print("-" * 40)
print('')
print('Q11. 事件循环模式:')
print('  事件循环是一种持续运行的循环模式：系统持续监听事件→处理→输出→回到监听状态。')
print('  适合场景：聊天机器人（等待消息→回复→等待下一条消息）、')
print('  实时监控系统（持续监听指标→异常时告警→继续监听）。')
print('  核心特点是"永远不退出"，系统长期驻留运行。')
print('')
print('Q12. 会议室预约助手工作流:')
print('  触发方式: 用户通过微信/飞书发送"预约XX会议室"')
print('  工具链:')
print('    1. intent_detector - 识别预约意图')
print('    2. calendar_query - 查询会议室可用时间')
print('    3. calendar_book - 执行预约')
print('    4. notification - 发送确认通知')
print('  输出通道: 微信确认消息 + 邮件日历邀请')
print('  错误处理:')
print('    - 会议室已满 → 推荐备选时间段')
print('    - 冲突 → 通知冲突的预约者')
print('    - 网络错误 → 重试3次后通知人工')
print('')


---

## 📖 英文术语表

| 术语 | 英文 | 简要说明 |
|------|------|----------|
| 工具编排 | **Tool Orchestration** | 管理多个工具的调用顺序、条件和依赖关系 |
| 定时任务 | **Cron Job** | 按Cron表达式定时自动执行的任务 |
| 消息路由 | **Message Routing** | 根据规则将消息分发到目标平台 |
| 工作流 | **Workflow** | 有组织的多步骤任务执行流程 |
| 事件驱动 | **Event-Driven** | 系统行为由事件触发而非主动轮询 |
| Webhook | **Webhook** | 外部系统在事件发生时主动推送通知的机制 |
| 事件总线 | **Event Bus** | 接收、过滤和分发事件的中央枢纽 |
| 扇出-汇聚 | **Fan-out/Fan-in** | 并行执行多个任务后合并结果的模式 |
| 幂等性 | **Idempotency** | 重复执行不产生副作用（安全重试） |
| 死信队列 | **Dead Letter Queue (DLQ)** | 存储处理失败事件的队列，支持重试 |


---

## 📝 本日总结

### 今日核心知识

```
1. 工具编排 = 从单工具到多工具链的自动化管理
   └─ 五个层次：L0单工具 → L1顺序 → L2条件 → L3并行 → L4事件循环

2. Cron定时 = 让Agent在指定时间自动执行任务
   └─ 表达式语法：分 时 日 月 周

3. 消息路由 = 多平台消息的智能分发
   └─ 四种策略：来源回传 / 规则路由 / 多通道广播 / 智能选择

4. 工作流设计 = 任务执行的蓝图
   └─ 四种模式：线性管道 / 扇出-汇聚 / 条件分支 / 事件循环

5. 事件驱动 = 系统自动响应外部变化
   └─ 事件源 → 事件总线 → 过滤 → 处理 → 推送

6. Webhook = 让外部系统主动通知Agent
   └─ 注册URL → 接收事件 → 验证签名 → 处理 → 响应
```

### 与往期知识的关联

| 往期内容 | 今天的关联 |
|---------|-------------|
| W4 RAG | 搜索工具 + 知识库 → 事件驱动的知识检索 |
| W5 Function Calling | 单工具调用 → 多工具编排的起点 |
| W6 Agent框架 | ReAct循环 → 工作流中的LLM决策节点 |
| Day1 System Prompt | 定义Agent能力 → 工具编排的"大脑配置" |

---

## 🔮 明天预告：Day3 — 多Agent协作

明天我们将学习：
- **Agent间通信**：Agent如何互相调用和协作
- **角色分工**：规划者Agent、执行者Agent、审核者Agent
- **任务分解**：复杂任务如何拆分给多个Agent
- **共享上下文**：Agent之间如何共享信息和状态
- **OpenClaw多Agent实践**：Subagent模式与TaskFlow

> 🎯 **思考题（为明天准备）**：
> 如果你要让两个Agent协作完成"写一篇技术博客"的任务，
> 你会如何分工？每个Agent负责什么？
